In [1]:
!pip -q install opencv-python==4.11.0.86

    kubernetes (>=9.0.0a1.0) ; extra == 'all_extras'
               ~~~~~~~~~~^


In [2]:
author = 'nm'
site = 'viti-levu'
year = '2025'

version = f"{author}-{site}-{year}"
version

'nm-viti-levu-2025'

In [3]:
from pystac_client import Client
import geopandas as gpd
import xarray as xr
import folium
from odc.geo import Geometry
from odc.stac import load
from odc.geo.xr import write_cog
from src import util
import rasterio as rio

In [4]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

In [5]:
year = "2025"
aoi = Geometry(gpd.read_file("aoi/fiji-180.geojson").geometry[0], crs="EPSG:4326")
aoi.explore()

In [6]:
inland = gpd.read_file("inland_800m.geojson")

In [7]:
# # elevation mask (30-50m)
# data["mangroves"], elevation_mask = mask_elevation(
# data["mangroves"], threshold=30, return_mask=True

In [8]:
items = list(
    client.search(collections=["dep_s2_geomad"], datetime=year, intersects=aoi).items()
)

print(f"Found {len(items)} items")

Found 20 items


In [9]:
data = load(
    items,
    measurements=[
        "nir",
        "red",
        "blue",
        "green",
        "green",
        "swir16",
    ],
    intersects=aoi,
    resolution=10,
    chunks={"x": 2048, "y": 2048},
    groupby="solar_day",
)

data

<xarray.Dataset> Size: 13GB
Dimensions:      (y: 37514, x: 34260, time: 1)
Coordinates:
  * y            (y) float64 300kB -1.802e+06 -1.802e+06 ... -2.178e+06
  * x            (x) float64 274kB 2.988e+06 2.988e+06 ... 3.331e+06 3.331e+06
    spatial_ref  int32 4B 3832
  * time         (time) datetime64[ns] 8B 2025-01-01
Data variables:
    nir          (time, y, x) uint16 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) uint16 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    blue         (time, y, x) uint16 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) uint16 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    swir16       (time, y, x) uint16 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>

In [10]:
# scale
data = (data.where(data != 0) * 0.0001).clip(0.0001, 1)

In [11]:
# input elevation 
elevation_threshold = 20
     

In [12]:
e84_catalog = "https://earth-search.aws.element84.com/v1/"
e84_client = Client.open(e84_catalog)
collection = "cop-dem-glo-30"

items_z = e84_client.search(
    collections=[collection],
    bbox=list(data.odc.geobox.geographic_extent.boundingbox)
).item_collection()


In [13]:
items_z = e84_client.search(
    collections=[collection],
    bbox=list(data.odc.geobox.geographic_extent.boundingbox)
).item_collection()

elevation_ds = load(
    items_z,
    measurements=["data"], 
    geobox=data.odc.geobox
)

elevation = elevation_ds.data.squeeze()

In [14]:
# elevation.plot(vmin=0, vmax=200, cmap="viridis")

In [15]:
elevation = elevation.where(elevation < elevation_threshold)
# elevation.plot(vmin=0, vmax=200, cmap="viridis")

In [16]:
mask = elevation <= elevation_threshold

data_masked = data.where(mask)

# data_masked.nir.isel(time=0).plot(robust=True)

In [17]:
data_masked["mndwi"] = (data_masked["green"] - data_masked["swir16"]) / (data_masked["green"] + data_masked["swir16"])

mndwi_threshold = -0.2

mndwi_mask = data_masked.mndwi < mndwi_threshold

# mndwi_mask.odc.explore()

In [18]:
data_masked = data.where(mndwi_mask)
data_masked
# data_masked.nir.isel(time=0).plot(robust=True)

<xarray.Dataset> Size: 26GB
Dimensions:      (time: 1, y: 37514, x: 34260)
Coordinates:
  * y            (y) float64 300kB -1.802e+06 -1.802e+06 ... -2.178e+06
  * x            (x) float64 274kB 2.988e+06 2.988e+06 ... 3.331e+06 3.331e+06
    spatial_ref  int32 4B 3832
  * time         (time) datetime64[ns] 8B 2025-01-01
Data variables:
    nir          (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    blue         (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    swir16       (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>

In [19]:
from rasterio import features
from rasterio.transform import from_bounds
import xarray as xr
import numpy as np

# Get the transform from your data's spatial coordinates
# Extract bounds from your coordinates
x_min = float(data_masked.x.min())
x_max = float(data_masked.x.max())
y_min = float(data_masked.y.min())
y_max = float(data_masked.y.max())

# Calculate pixel size (resolution)
x_res = float(data_masked.x[1] - data_masked.x[0])
y_res = float(data_masked.y[1] - data_masked.y[0])

# Create the affine transform
transform = from_bounds(x_min, y_min, x_max, y_max, 
                       data_masked.sizes['x'], data_masked.sizes['y'])

# Ensure inland GeoDataFrame is in the same CRS as your data (EPSG:3832)
inland_reprojected = inland.to_crs(3832)

# Rasterize the inland geometries
mask_array = features.rasterize(
    [(geom, 1) for geom in inland_reprojected.geometry],
    out_shape=(data_masked.sizes['y'], data_masked.sizes['x']),
    transform=transform,
    fill=0,
    all_touched=True
)

# Create a DataArray with only spatial dimensions
mask_da = xr.DataArray(
    mask_array,
    coords={'y': data_masked.y, 'x': data_masked.x},
    dims=('y', 'x')
)

# Apply mask to entire dataset (broadcasts across time automatically)
data_masked_clipped = data_masked.where(mask_da == 0)

print(f"Clipped data shape: {data_masked_clipped.dims}")

Clipped data shape: FrozenMappingWarningOnValuesAccess({'time': 1, 'y': 37514, 'x': 34260})


In [20]:
# data_masked_clipped.nir.plot()
data_masked = data_masked_clipped

### AMMI

Automatic Mangrove Map and Index (AMMI)
Suyarso (2022) developed a mangrove vegetation index that
simultaneously extracts mangroves and computes canopy density
precisely using optical satellite imagery, e.g., Sentinel-2 and
Landsat-5, Landsat-7, and Landsat-8. The algorithm is the
product of two equations. The first equation should separate the
land and vegetation from water features. The second equation
should map the extent of mangroves and display the canopy
density. The proponent did not provide a threshold range but
from his results, it was between 5 to 10.

In [21]:
nir = data_masked["nir"]
swir = data_masked["swir16"]
red = data_masked["red"]
green = data_masked["green"]
blue = data_masked["blue"]

ammi = ((nir - red) / (red + swir)) * ((nir - swir) / (swir - 0.65 * red))

ammi = ammi.to_dataset(name="ammi")
   
ammi

<xarray.Dataset> Size: 5GB
Dimensions:      (y: 37514, x: 34260, time: 1)
Coordinates:
  * y            (y) float64 300kB -1.802e+06 -1.802e+06 ... -2.178e+06
  * x            (x) float64 274kB 2.988e+06 2.988e+06 ... 3.331e+06 3.331e+06
    spatial_ref  int32 4B 3832
  * time         (time) datetime64[ns] 8B 2025-01-01
Data variables:
    ammi         (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>

Spatially, the AMMI captures the mangroves from sparse mangroves, indicated by the low spectral sensitivity with an index of about 5, to dense mangroves (>20), and the index below 5 is classified as nonmangrove.

https://onlinelibrary.wiley.com/doi/10.1155/2022/8103242

In [22]:

# AMMI_THRESHOLD = 4.0 - 20
AMMI_THRESHOLD = range(4, 20)

mangrove_mask = ammi.ammi >= list(AMMI_THRESHOLD)[4]

num_vals = len(AMMI_THRESHOLD)
for i, val in enumerate(AMMI_THRESHOLD, 1):
    # print(f"{i}: {val}")
    density_percentage = 10 + (i - 1) * (90 / (num_vals - 1))
    mangrove_mask = xr.where(ammi.ammi >= val, density_percentage, mangrove_mask)
    

# Convert boolean mask to uint8 (0/1) and attach geospatial metadata for saving
mangrove_mask = mangrove_mask.astype("uint8")
mangrove_mask = mangrove_mask.compute()

# remove null
data_masked["mangroves"] = mangrove_mask.where(mangrove_mask != 0, drop=True)
data_masked["mangroves"]

<xarray.DataArray 'mangroves' (time: 1, y: 37514, x: 34260)> Size: 5GB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], dtype=float32)
Coordinates:
  * y            (y) float64 300kB -1.802e+06 -1.802e+06 ... -2.178e+06
  * x            (x) float64 274kB 2.988e+06 2.988e+06 ... 3.331e+06 3.331e+06
  * time         (time) datetime64[ns] 8B 2025-01-01
    spatial_ref  int32 4B 3832

/srv/conda/envs/notebook/lib/python3.11/site-packages/odc/geo/_rgba.py:55: RuntimeWarning: invalid value encountered in cast
  return x.astype("uint8")


### Apply Morphological Filter

In [23]:
CLOSING_KERNEL_SIZE = 2
OPENING_KERNEL_SIZE = 2

In [24]:
# from scipy import ndimage  # <--- Add this line
# import numpy as np

# def filter_mask(mask, closing_kernel_size=3, opening_kernel_size=6):
#     # 1. Squeeze and ensure 2D
#     data = mask.values.squeeze() if hasattr(mask, 'values') else mask.squeeze()
    
#     # Define kernels
#     struct_closing = np.ones((closing_kernel_size, closing_kernel_size))
#     struct_opening = np.ones((opening_kernel_size, opening_kernel_size))

#     # 2. Apply morphological operations
#     # Now 'ndimage' will be recognized
#     step1 = ndimage.binary_closing(data, structure=struct_closing)
#     step2 = ndimage.binary_opening(step1, structure=struct_opening)
    
#     # 3. Convert to Float and set False to NaN for transparency
#     result = step2.astype(float)
#     result[result == 0] = np.nan
    
#     # 4. Return with original metadata
#     if hasattr(mask, 'values'):
#         return mask.copy(data=result.reshape(mask.shape))
    
#     return result

In [25]:
# data_masked['mangroves'] = filter_mask(closing_kernel_size=CLOSING_KERNEL_SIZE, opening_kernel_size=OPENING_KERNEL_SIZE, mask=data['mangroves'])

In [26]:
# data_masked["mangroves"].plot()

In [27]:
# Count pixels that are classified as mangroves
mangrove_pixel_count = (data_masked["mangroves"] > 0).sum().item()
print(f"Number of mangrove pixels: {mangrove_pixel_count}")

Number of mangrove pixels: 5292578


In [28]:
total_pixels = data_masked.mangroves.size
print(f"Total grid pixels: {total_pixels}")

Total grid pixels: 1285229640


### Visual Verification

In [29]:
gdf = gpd.read_parquet("gmw_v4_pacific_validation.parquet")
aoi = aoi.to_crs(3832)
gdf = gdf.clip(aoi.geom, keep_geom_type=False)
gdf.COUNTRY.unique()

array(['Fiji'], dtype=object)

In [30]:
# Compute the RGB into memory
rgb_ds = data.odc.to_rgba(vmin=0, vmax=0.3, bands=("red", "green", "blue")).compute()

In [31]:
# # Initialize the Folium map centered on the average centroid
# x, y = aoi.centroid.to_crs("epsg:4326").xy
# m = folium.Map(location=[float(y[0]), float(x[0])], zoom_start=10)

# # GMW for Validation
# # gdf.geometry.boundary.explore(m=m, color="red", name="GMWv4")
# gdf.explore(m=m, 
#             name="GMWv4",
#             style_kwds=dict(
#                 fillOpacity=0.5,   # Sets the opacity of the fill color
#                 opacity=1.0,       # Sets the opacity of the outline color
#                 color="red",       # Outline color
#                 fillColor="green") # Fill color
#             )

# # RGB for Comparison
# rgb_ds.odc.add_to(m, name="GeoMAD True Color")

# # Mangrove Mask and Density
# data_masked["mangroves"].odc.add_to(m, cmap="magma", name="AMMI")

# folium.LayerControl().add_to(m)

# # m

In [32]:
import matplotlib.pyplot as plt

# # 1. Setup the transparent colormap
# current_cmap = plt.get_cmap('Greens').copy() 
# current_cmap.set_bad(alpha=0) # This makes NaNs transparent

# # 2. Plot with explicit range limits
# data_masked['mangroves'].plot(
#     cmap=current_cmap, 
#     vmin=0,         # Minimum range
#     vmax=1,         # Maximum range (where your mangroves are)
#     add_colorbar=False
# )

# plt.title("Filtered Mangrove Mask: Suva Region")
# plt.show()

In [33]:
data_masked

<xarray.Dataset> Size: 31GB
Dimensions:      (y: 37514, x: 34260, time: 1)
Coordinates:
  * y            (y) float64 300kB -1.802e+06 -1.802e+06 ... -2.178e+06
  * x            (x) float64 274kB 2.988e+06 2.988e+06 ... 3.331e+06 3.331e+06
  * time         (time) datetime64[ns] 8B 2025-01-01
    spatial_ref  int32 4B 3832
Data variables:
    nir          (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    blue         (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    swir16       (time, y, x) float32 5GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    mangroves    (time, y, x) float32 5GB nan nan nan nan ... nan nan nan nan

### Export

In [34]:
data_masked.mangroves.odc.write_cog(f"ammi-1.2-{version}.tif", overwrite=True)

PosixPath('ammi-1.2-nm-viti-levu-2025.tif')